In [15]:
import pandas as pd

from utils.data_processing import (
    transform_counts_data,
    redefined_taxa_short_name,
    dict_ASV_names,
    otuID_taxonomy,
)

from utils.plotting_functions import do_partial_correlations

## Load the stool quality metadata

In [16]:
stool_quality_meta = pd.read_csv("../../data/stool_quality_meta.csv", index_col=0)

## Load the counts data

In [17]:
path_counts_data = "../../qiime/table_rarefied.tsv"
counts_data = pd.read_csv(path_counts_data, sep="\t", index_col=0, skiprows=1)

# counts_data.index = counts_data.index.map(lambda i: otuID_taxonomy.loc[i].values[0].replace("; ",";")) ## rename ASV id to taxon
# counts_data = counts_data.groupby(counts_data.index).sum() ## sum over ASVs with same taxon
counts_data = counts_data.T

counts_data.index.name = "sample-id"

transformed_counts_data = transform_counts_data(counts_data, method="clr")

### Optional: Run direct correlations

In [18]:
# corr_to_great_stool_quality = (
#     stool_quality_meta.merge(
#         transformed_counts_data, left_on="sample-id", right_index=True, how="inner"
#     )
#     .corr(method="spearman", numeric_only=True)["great_stool_quality_proportion"]
#     .sort_values(ascending=False)
#     .drop(
#         [
#             "great_stool_quality_proportion",
#             "diarrhea_proportion",
#             "qs_great",
#             "qs_normal",
#             "qs_constipated",
#             "qs_diarrhea",
#         ]
#     )
#     .dropna()
# )

# corr_to_diarrhea = (
#     stool_quality_meta.merge(
#         transformed_counts_data, left_on="sample-id", right_index=True, how="inner"
#     )
#     .corr(method="spearman", numeric_only=True)["diarrhea_proportion"]
#     .sort_values(ascending=False)
#     .drop(
#         [
#             "great_stool_quality_proportion",
#             "diarrhea_proportion",
#             "qs_great",
#             "qs_normal",
#             "qs_constipated",
#             "qs_diarrhea",
#         ]
#     )
#     .dropna()
# )

## Run partial correlations

In [19]:
parcorr_microbe_to_diarrhea = do_partial_correlations(
    transformed_counts_data,
    stool_quality_meta.set_index("sample-id"),
    "diarrhea_proportion",
    covariates=["age", "bmi"],
    corr_threshold=0.15,
    method="spearman",
).sort_values("r")

In [20]:
parcorr_microbe_to_diarrhea["redefined_short_name"] = parcorr_microbe_to_diarrhea[
    "microbe"
].map(
    lambda i: redefined_taxa_short_name(
        otuID_taxonomy.loc[i].values[0].replace("; ", ";"), level="genus"
    )
)
parcorr_microbe_to_diarrhea["ASV_name"] = parcorr_microbe_to_diarrhea["microbe"].map(
    lambda i: dict_ASV_names[i]
)

parcorr_microbe_to_diarrhea["short_name"] = (
    parcorr_microbe_to_diarrhea["ASV_name"]
    + " ("
    + parcorr_microbe_to_diarrhea["redefined_short_name"]
    + ")"
)

parcorr_microbe_to_diarrhea.rename(columns={"r": "Diarrhea Proportion"}, inplace=True)

## Save the partial correlations to data folder

In [21]:
parcorr_microbe_to_diarrhea.to_csv(
    "../../data/partial_correlations_microbe_to_diarrhea.csv"
)